# SarcAsM quickstart

To run the examples, extract the zipped folder `test_data` into the project root.

Each recording is a `SarcAsM` object. Its results are stored in `<name>.ome.zarr` next to the TIFF and are read
through `sarc.data`; a key is its path, so `sarc.data['structure.sarcomere.oop']` is
`sarc.data.structure.sarcomere.oop`. `print(sarc)` summarises the recording, `sarc.data.find('oop')` searches the
keys, and `help(sarc.structure.analyze_z_bands)` shows the parameters of a step.

Documentation and detailed tutorials: [SarcAsM docs](https://filedn.eu/lKfS794F9UgX7PDuBQcfChB/SarcAsM_docs/).

In [ ]:
from sarcasm import *
import matplotlib.pyplot as plt

## Structure analysis

In [ ]:
# a single image or a movie
sarc = SarcAsM('./test_data/long_term_2D_ACTN2-citrine_CM/20211115_ACTN2_CMs_96well_control_12days.tif')

### Detection of Z-bands, M-bands, orientation and cell mask by deep learning

In [ ]:
sarc.detect_sarcomeres(max_patch_size=(2048, 2048))

In [ ]:
frame = 33
zoom_region = (1500, 1800, 800, 1100)
inset_bounds = (0.05, 0.05, 0.4, 0.4)

fig, axs = plt.subplots(2, 3, figsize=(12, 8.5), constrained_layout=True)
Plots.plot_image(axs[0, 0], sarc, frame=frame, title='Image', zoom_region=zoom_region, inset_bounds=inset_bounds)
Plots.plot_z_bands_midlines(axs[0, 1], sarc, frame=frame, title='Z-bands and M-bands', zoom_region=zoom_region,
                            inset_bounds=inset_bounds)
Plots.plot_sarcomere_mask(axs[0, 2], sarc, frame=frame, title='Sarcomere mask', zoom_region=zoom_region,
                          inset_bounds=inset_bounds)
Plots.plot_sarcomere_orientation_field(axs[1, 0], axs[1, 1], sarc, frame=frame, orient_colorbar='horizontal',
                                       zoom_region=zoom_region, inset_bounds=inset_bounds)
Plots.plot_image(axs[1, 2], sarc, frame=frame)
Plots.plot_cell_mask(axs[1, 2], sarc, frame=frame, title='Cell mask', alpha=0.4, scalebar=False)
plt.show()

### Z-bands, sarcomere vectors, myofibrils and domains

Here for one frame; leave out `frames` to analyse all. `sarc.structure.full_analysis()` runs all four steps with
default parameters.

In [ ]:
sarc.structure.analyze_z_bands(frames=frame)
sarc.structure.analyze_vectors(frames=frame)
sarc.structure.analyze_myofibrils(frames=frame)
sarc.structure.analyze_domains(frames=frame)

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(12, 8.5), constrained_layout=True)
Plots.plot_z_segmentation(axs[0, 0], sarc, frame=frame, title='Z-band segmentation', zoom_region=zoom_region,
                          inset_bounds=inset_bounds)
Plots.plot_z_lateral_connections(axs[0, 1], sarc, frame=frame, title='Z-band lateral connections',
                                 zoom_region=zoom_region, inset_bounds=inset_bounds)
Plots.plot_sarcomere_vectors(axs[0, 2], sarc, frame=frame, s_points=0.1, title='Sarcomere vectors',
                             zoom_region=zoom_region, inset_bounds=inset_bounds)
Plots.plot_myofibril_lines(axs[1, 0], sarc, frame=frame, title='Myofibril lines', zoom_region=zoom_region,
                           inset_bounds=inset_bounds)
Plots.plot_myofibril_length_map(axs[1, 1], sarc, frame=frame, title='Myofibril length', orient_colorbar='horizontal',
                                zoom_region=zoom_region, inset_bounds=inset_bounds)
Plots.plot_sarcomere_domains(axs[1, 2], sarc, frame=frame, title='Sarcomere domains')
plt.show()

print(f"mean sarcomere length {sarc.data['structure.sarcomere.slen_mean'][frame]:.2f} µm, "
      f"OOP {sarc.data['structure.sarcomere.oop'][frame]:.2f}")

## Motion analysis

### Tracking of individual sarcomeres

In [ ]:
# a high-speed movie of a beating cardiomyocyte
sarc = SarcAsM('./test_data/high_speed_single_ACTN2-citrine_CM/30kPa.tif')

# detection in every frame, plus the less flickering Z-bands of the 3D U-Net for high-speed movies
sarc.detect_sarcomeres()
sarc.motion.detect_z_bands_fast_movie()

# sarcomere vectors in every frame, then track them through the movie
sarc.structure.analyze_vectors()
sarc.motion.track()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
Plots.plot_tracks(ax, sarc, color_by='coverage', show_z_bands=True, colorbar=True, title='Sarcomere tracks')
plt.show()

### Grouped motion analysis

The tracks are grouped (here per myofibril; also `'pool'`, `'mband'`, `'loi'`, `'domain'` or `'custom'`) and the
contractions of each group are analysed. Each group `g` gives its results as attributes (`g.beating_rate`,
`g.slen_avg`, ...) and its member tracks as `g.tracks.slen`, `g.tracks.delta_slen`, ...

In [ ]:
sarc.structure.analyze_myofibrils(frames=0)
myofibrils = sarc.motion.analyze('myofibril', reference_frame=0, min_group_size=6)
g = max(myofibrils, key=lambda g: g.n_members)
print(f'{len(myofibrils)} myofibrils; myofibril {g.id}: {g.n_members} sarcomeres, beating rate {g.beating_rate:.2f} Hz')

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3))
Plots.plot_track_groups(axs[0], sarc, 'myofibril', show_z_bands=True, title='Myofibril groups')
Plots.plot_slen(axs[1], g, t_lim=(0, 5))
plt.show()

In [ ]:
Plots.plot_group_summary(g, t_lim=(0, 5))
plt.show()

### Export

In [ ]:
Export.export_motion_data(sarc, './motion_myofibril.xlsx', kind='myofibril')